# Phase 2 Validation: Collection Health + Embedding Integrity

In [1]:
import os

os.environ["HF_HOME"] = "D:/hf_cache"
os.environ["HF_HUB_CACHE"] = "D:/hf_cache/hub"

import json
import logging
import random
from collections import defaultdict
from pathlib import Path

from qdrant_client import QdrantClient

In [2]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("validate_embeddings")

PAGE_METADATA_FILE = Path("D:/Self Learning/DEPI/R4/DEPI Project/Data/extracted/Page Images/metadata/page_metadata.jsonl")
COLLECTION_NAME = "depi_page_images"
VECTOR_DIM = 128
SAMPLE_SIZE = 30  # number of points to deep-check (with actual vectors)

In [3]:
def get_client() -> QdrantClient:
    return QdrantClient(url="http://localhost:6333", timeout=60)

### Check 1: Collection-level health

In [4]:
def check_collection_health(client: QdrantClient) -> dict:
    info = client.get_collection(COLLECTION_NAME)

    status = info.status
    points_count = info.points_count
    indexed_vectors_count = info.indexed_vectors_count
    segments_count = info.segments_count

    print(f"\n{'='*60}")
    print("COLLECTION HEALTH")
    print(f"{'='*60}")
    print(f"Status:                 {status}")
    print(f"Points count:           {points_count}")
    print(f"Indexed vectors count:  {indexed_vectors_count}")
    print(f"Segments count:         {segments_count}")

    if str(status).lower() == "grey":
        print("\n⚠️  Status is GREY — optimizations pending but paused.")
        print("    This is not data loss. Click 'Trigger Optimizers' in the")
        print("    dashboard, or this script will send a harmless no-op update")
        print("    below to nudge it.")

    return {
        "status": str(status),
        "points_count": points_count,
        "indexed_vectors_count": indexed_vectors_count,
        "segments_count": segments_count,
    }
    
    
def trigger_optimizers_if_needed(client: QdrantClient, health: dict) -> None:
    """Sends a no-op config update — any update operation is enough to
    wake a paused (grey) optimizer, per Qdrant's own documented behavior."""
    if health["status"].lower() != "grey":
        return

    logger.info("Sending no-op update to trigger paused optimizers...")
    client.update_collection(collection_name=COLLECTION_NAME, optimizer_config={})
    logger.info("Update sent. Re-check the dashboard in a minute — status should turn yellow, then green.")

### Check 2: Per-document completeness — the check that a raw point

In [5]:
def load_expected_pages() -> dict[str, int]:
    """doc_id -> expected total_pages, from Phase 1's ground truth."""
    expected = {}
    with open(PAGE_METADATA_FILE, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            expected[record["doc_id"]] = record["total_pages"]
    return expected


def scan_indexed_pages(client: QdrantClient) -> dict[str, set[int]]:
    """doc_id -> set of page_numbers actually present in Qdrant.
    Payload-only scroll — cheap, no vectors pulled."""
    indexed = defaultdict(set)
    offset = None

    while True:
        points, offset = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=1000,
            with_payload=["doc_id", "page_number"],
            with_vectors=False,
            offset=offset,
        )
        for point in points:
            indexed[point.payload["doc_id"]].add(point.payload["page_number"])

        if offset is None:
            break

    return indexed


def check_document_completeness(client: QdrantClient) -> dict:
    print(f"\n{'='*60}")
    print("PER-DOCUMENT COMPLETENESS")
    print(f"{'='*60}")

    expected = load_expected_pages()
    indexed = scan_indexed_pages(client)

    incomplete_docs = []
    for doc_id, expected_total in expected.items():
        indexed_pages = indexed.get(doc_id, set())
        if len(indexed_pages) != expected_total:
            missing = set(range(1, expected_total + 1)) - indexed_pages
            incomplete_docs.append({
                "doc_id": doc_id,
                "expected": expected_total,
                "indexed": len(indexed_pages),
                "missing_pages": sorted(missing),
            })

    total_docs = len(expected)
    complete_docs = total_docs - len(incomplete_docs)

    print(f"Total documents:        {total_docs}")
    print(f"Fully indexed:          {complete_docs}")
    print(f"Incomplete:             {len(incomplete_docs)}")

    if incomplete_docs:
        print("\n⚠️  Documents with missing pages:")
        for doc in incomplete_docs[:20]:  # cap printed output
            print(f"   - {doc['doc_id']}: {doc['indexed']}/{doc['expected']} pages, missing {doc['missing_pages']}")
        if len(incomplete_docs) > 20:
            print(f"   ... and {len(incomplete_docs) - 20} more")
    else:
        print("\n✅ Every document has all expected pages indexed.")

    return {"total_docs": total_docs, "complete_docs": complete_docs, "incomplete_docs": incomplete_docs}

### Check 3: Vector integrity — sample-based deep check

In [6]:
def check_vector_integrity(client: QdrantClient) -> dict:
    print(f"\n{'='*60}")
    print(f"VECTOR INTEGRITY (sample of {SAMPLE_SIZE})")
    print(f"{'='*60}")

    # Pull a random sample of real point IDs directly from the collection
    sample_points, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        limit=SAMPLE_SIZE,
        with_payload=True,
        with_vectors=False,
    )
    sample_ids = [p.id for p in sample_points]

    # Now fetch those specific points WITH vectors for a deep check
    detailed_points = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=sample_ids,
        with_payload=True,
        with_vectors=True,
    )

    passed, failed = 0, 0
    failures = []

    for point in detailed_points:
        issues = []

        vectors = point.vector
        if not vectors or len(vectors) == 0:
            issues.append("empty vector list")
        else:
            wrong_dim = [v for v in vectors if len(v) != VECTOR_DIM]
            if wrong_dim:
                issues.append(f"{len(wrong_dim)} sub-vectors have wrong dimensionality")

            all_zero = all(all(x == 0.0 for x in v) for v in vectors)
            if all_zero:
                issues.append("all vectors are zero — likely a silent embedding failure")

            has_nan = any(any(x != x for x in v) for v in vectors)  # NaN != NaN
            if has_nan:
                issues.append("contains NaN values")

        image_path = point.payload.get("image_path")
        if not image_path or not Path(image_path).exists():
            issues.append(f"image file missing on disk: {image_path}")

        required_keys = {"doc_id", "doc_type", "page_number", "total_pages", "image_path", "source_pdf"}
        missing_keys = required_keys - set(point.payload.keys())
        if missing_keys:
            issues.append(f"missing payload keys: {missing_keys}")

        if issues:
            failed += 1
            failures.append({"id": point.id, "doc_id": point.payload.get("doc_id"), "issues": issues})
        else:
            passed += 1

    print(f"Passed:  {passed}/{len(detailed_points)}")
    print(f"Failed:  {failed}/{len(detailed_points)}")

    if failures:
        print("\n⚠️  Issues found:")
        for f in failures:
            print(f"   - {f['doc_id']} (id={f['id']}): {', '.join(f['issues'])}")
    else:
        print("\n✅ All sampled points have valid vectors and complete payloads.")

    return {"passed": passed, "failed": failed, "failures": failures}

### Run

In [7]:
if __name__ == "__main__":
    client = get_client()
    try:
        health = check_collection_health(client)
        trigger_optimizers_if_needed(client, health)
        completeness = check_document_completeness(client)
        integrity = check_vector_integrity(client)

        print(f"\n{'='*60}")
        print("SUMMARY")
        print(f"{'='*60}")
        all_good = (
            completeness["incomplete_docs"] == []
            and integrity["failed"] == 0
        )
        if all_good:
            print("🏆 Phase 2 validation passed — safe to proceed to Phase 3.")
        else:
            print("⚠️  Issues found above — review before proceeding to Phase 3.")
    finally:
        client.close()

2026-07-03 15:39:31,267 | INFO | HTTP Request: GET http://localhost:6333/collections/depi_page_images "HTTP/1.1 200 OK"
2026-07-03 15:39:31,269 | INFO | Sending no-op update to trigger paused optimizers...
2026-07-03 15:39:31,282 | INFO | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"



COLLECTION HEALTH
Status:                 grey
Points count:           22664
Indexed vectors count:  16008
Segments count:         17

⚠️  Status is GREY — optimizations pending but paused.
    This is not data loss. Click 'Trigger Optimizers' in the
    dashboard, or this script will send a harmless no-op update
    below to nudge it.


2026-07-03 15:39:33,309 | INFO | HTTP Request: PATCH http://localhost:6333/collections/depi_page_images "HTTP/1.1 200 OK"
2026-07-03 15:39:33,311 | INFO | Update sent. Re-check the dashboard in a minute — status should turn yellow, then green.



PER-DOCUMENT COMPLETENESS


2026-07-03 15:39:35,439 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:37,524 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:39,566 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:41,622 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:43,650 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:45,712 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:47,775 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:39:49,807 | INFO | HTTP Request: POST htt

Total documents:        380
Fully indexed:          380
Incomplete:             0

✅ Every document has all expected pages indexed.

VECTOR INTEGRITY (sample of 30)


2026-07-03 15:40:22,701 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points/scroll "HTTP/1.1 200 OK"
2026-07-03 15:40:24,934 | INFO | HTTP Request: POST http://localhost:6333/collections/depi_page_images/points "HTTP/1.1 200 OK"


Passed:  30/30
Failed:  0/30

✅ All sampled points have valid vectors and complete payloads.

SUMMARY
🏆 Phase 2 validation passed — safe to proceed to Phase 3.
